# QUERCUS v8 — Demo Notebook
**Quantitative Unsupervised Extraction and Remote Classification of Unstructured Scenes**

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github.com/NASA-EarthRISE/quercus/blob/main/QUERCUS_v8_demo.ipynb)

This notebook demonstrates the QUERCUS Python package end-to-end.

| Pipeline | Steps |
|----------|-------|
| **1984 Aerials** | Download selected frames -> clip borders -> georeference -> SAM -> NDVI -> CSV -> mosaic |
| **NAIP** | Fetch from GEE matching bbox -> SAM -> NDVI -> CSV -> mosaic |
| **Classify** | K-Means -> oak extraction |

**Before running:** Runtime -> Change runtime type -> **T4 GPU**

## 1. Install QUERCUS
Run once. After completion: **Runtime -> Restart runtime**.

In [ ]:
# Install QUERCUS directly from GitHub
!pip install -q git+https://github.com/NASA-EarthRISE/quercus.git
!pip install -q segment-geospatial
!pip install -q earthengine-api folium

import quercus
print(f'QUERCUS v{quercus.__version__} installed successfully')

## 2. Keep-alive (prevent overnight disconnect)

In [ ]:
from IPython.display import Javascript
display(Javascript('''
function clickConnect(){
    console.log('Keeping alive...');
    document.querySelector('colab-toolbar-button#connect').click()
}
setInterval(clickConnect, 60000)
'''))
print('Keep-alive active')

## 3. Authenticate Google Earth Engine

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)  # set GEE_PROJECT in Config below

## 4. Configuration
Edit this cell before running the pipeline.

In [ ]:
# ── Specific frame selection ────────────────────────────────────────────────
# Canvas indices (cv= values from Berkeley viewer URL)
# https://digicoll.lib.berkeley.edu/record/312848?v=uv#?cv=715
RECORD_ID = '312848'
PAGE_IDS  = [715, 716, 717, 718, 1104, 1105, 1670, 1671, 1672, 1673, 1674]

# ── AOI (Area of Interest) ───────────────────────────────────────────────────
# Frames outside this area will be skipped and reported.
# Options:
#   None                    -- no filtering, download all requested frames
#   'my_study_area.geojson' -- path to a GeoJSON file
#   shapely geometry object -- any Shapely polygon/multipolygon
AOI = None

# ── Survey anchor (for estimated GCPs) ──────────────────────────────────────
# Approximate location of the survey block.
# For publication quality, replace with real GCPs from QGIS Georeferencer.
ANCHOR_LON       = -122.254   # western edge of frame 0
ANCHOR_LAT0      =  37.844    # south edge
ANCHOR_LAT1      =  37.852    # north edge
FRAME_WIDTH_DEG  =  0.008     # ~700m at 37N
FRAME_STEP_DEG   =  0.0056    # 70% step (30% overlap between frames)

# ── NAIP year range ──────────────────────────────────────────────────────────
# NAIP is not annual. California availability:
# 2009, 2012, 2014, 2016, 2018, 2020, 2022
# Adjust these if no images are found for your study area.
NAIP_YEAR_START = 2018
NAIP_YEAR_END   = 2022
NAIP_SCALE      = 1.0   # metres per pixel (1.0 = native NAIP resolution)

# ── GEE + SAM ───────────────────────────────────────────────────────────────
GEE_PROJECT = None        # e.g. 'my-gee-project'
SAM_MODEL   = 'vit_h'    # vit_h best quality; vit_b if RAM limited
NDVI_YEAR   = 1984        # Landsat NDVI year for 1984 aerials
KMEANS_K    = None        # None = auto elbow detection

print('Config set')
print(f'Frames: {PAGE_IDS}')
print(f'AOI: {AOI}')
print(f'NAIP years: {NAIP_YEAR_START}-{NAIP_YEAR_END}')

## 5. D1 — Download 1984 Aerial Frames
Frames outside the AOI are automatically skipped.

In [ ]:
from quercus.ingest import fetch_iiif_images

raw_paths = fetch_iiif_images(
    page_ids=PAGE_IDS,
    output_dir='data/raw',
    record_id=RECORD_ID,
    aoi=AOI,
    anchor_lon=ANCHOR_LON,
    anchor_lat0=ANCHOR_LAT0,
    anchor_lat1=ANCHOR_LAT1,
    frame_width_deg=FRAME_WIDTH_DEG,
    frame_step_deg=FRAME_STEP_DEG,
)
print(f'Downloaded {len(raw_paths)} frames')

## 6. Clip Scan Borders

In [ ]:
import warnings, logging, shutil
warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

from quercus.ingest import clip_scan_borders

shutil.rmtree('data/clipped', ignore_errors=True)
clipped_paths = clip_scan_borders(raw_paths, output_dir='data/clipped')
print(f'Clipped {len(clipped_paths)} images')

# Visual check
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
n = min(3, len(clipped_paths))
fig, axes = plt.subplots(1, n, figsize=(6*n, 5))
if n == 1: axes = [axes]
for i, ax in enumerate(axes):
    img  = np.array(Image.open(clipped_paths[i]).convert('RGB'))
    edge = np.mean([img[0].mean(),img[-1].mean(),img[:,0].mean(),img[:,-1].mean()])
    ax.imshow(img, cmap='gray')
    ax.set_title(f'{clipped_paths[i].name}\nedge={edge:.0f} [{"OK" if edge<160 else "BORDERS REMAIN"}]')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Georeference
> For publication-quality results, replace estimated GCPs with real ones from QGIS Georeferencer.

In [ ]:
import warnings, logging
warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

from quercus.ingest import georeference_images
from quercus.ingest.georeference import make_estimated_gcps

# Build estimated GCPs for each clipped image
# Replace with real GCPs from QGIS for accurate spatial placement:
# MY_GCPS = [
#     [(col, row, lon, lat), ...],  # image 0
#     [(col, row, lon, lat), ...],  # image 1
# ]
MY_GCPS = make_estimated_gcps(
    clipped_paths,
    anchor_lon=ANCHOR_LON,
    anchor_lat0=ANCHOR_LAT0,
    anchor_lat1=ANCHOR_LAT1,
    frame_width_deg=FRAME_WIDTH_DEG,
    frame_step_deg=FRAME_STEP_DEG,
)

georef_paths = georeference_images(
    clipped_paths,
    gcps_per_image=MY_GCPS,
    output_dir='data/georef',
    reference_lon=ANCHOR_LON,
)
print(f'Georeferenced {len(georef_paths)} images')

## 8. SAM Segmentation — 1984 Aerials

In [ ]:
from quercus.segment import run_sam_pipeline

aerial_results = run_sam_pipeline(
    georef_paths,
    output_dir='data/sam_outputs_1984',
    model_type=SAM_MODEL,
    points_per_side=64,
    pred_iou_thresh=0.50,
    stability_score_thresh=0.50,
    crop_n_layers=1,
    min_mask_region_area=50,
    max_px=1024,
    source_label='1984_aerial',
)
total_segs = sum(len(g) for g, _ in aerial_results)
print(f'1984 SAM: {total_segs:,} total segments across {len(aerial_results)} images')

## 9. Build 1984 Objects CSV + NDVI

In [ ]:
from quercus.segment import build_objects_csv
import pandas as pd

objects_csv_1984 = build_objects_csv(
    aerial_results,
    output_csv='data/outputs/quercus_objects_1984.csv',
    ndvi_year=NDVI_YEAR,
    gee_project=GEE_PROJECT,
    skip_ndvi=False,  # set True to skip GEE during testing
)
df_1984 = pd.read_csv(objects_csv_1984)
print(f'1984 CSV: {len(df_1984):,} rows')
df_1984.head()

## 10. Build 1984 Mosaic

In [ ]:
from quercus.ingest import build_mosaic

mosaic_1984 = build_mosaic(georef_paths, output_path='data/outputs/mosaic_1984.tif')
print(f'1984 mosaic: {mosaic_1984}')

## 11. NAIP Pipeline
Fetches NAIP from GEE matching each 1984 tile bbox. Tiles outside AOI are skipped.

In [ ]:
from quercus.segment import fetch_naip_and_run_sam

# NAIP California availability: 2009, 2012, 2014, 2016, 2018, 2020, 2022
# Adjust NAIP_YEAR_START/END in Config if no images found
naip_results = fetch_naip_and_run_sam(
    georef_paths,
    output_dir='data/naip_outputs',
    naip_year_start=NAIP_YEAR_START,
    naip_year_end=NAIP_YEAR_END,
    naip_scale=NAIP_SCALE,
    aoi=AOI,
    gee_project=GEE_PROJECT,
    model_type=SAM_MODEL,
    points_per_side=64,
    pred_iou_thresh=0.50,
    stability_score_thresh=0.50,
    crop_n_layers=1,
    min_mask_region_area=50,
    max_px=1024,
)
total_naip = sum(len(g) for g, _ in naip_results)
print(f'NAIP SAM: {total_naip:,} total segments across {len(naip_results)} tiles')

## 12. Build NAIP Objects CSV + NDVI

In [ ]:
from quercus.segment import build_objects_csv
import pandas as pd

# Determine NAIP year for NDVI composite
naip_years = [int(g['naip_year'].iloc[0]) for g,_ in naip_results
              if len(g) and 'naip_year' in g.columns]
naip_ndvi_year = max(set(naip_years), key=naip_years.count) if naip_years else 2020
print(f'Using NDVI year: {naip_ndvi_year}')

objects_csv_naip = build_objects_csv(
    naip_results,
    output_csv='data/outputs/quercus_objects_naip.csv',
    ndvi_year=naip_ndvi_year,
    gee_project=GEE_PROJECT,
    skip_ndvi=False,
)
df_naip = pd.read_csv(objects_csv_naip)
print(f'NAIP CSV: {len(df_naip):,} rows')
df_naip.head()

## 13. Build NAIP Mosaic

In [ ]:
from quercus.ingest import build_mosaic
from pathlib import Path

naip_tifs = sorted(Path('data/naip_outputs/naip_tifs').glob('*.tif'))
if naip_tifs:
    mosaic_naip = build_mosaic(naip_tifs, output_path='data/outputs/mosaic_naip.tif')
    print(f'NAIP mosaic: {mosaic_naip}')
else:
    print('No NAIP tiles to mosaic')
    mosaic_naip = None

## 14. Combined CSV (1984 + NAIP)

In [ ]:
import pandas as pd

df_combined  = pd.concat([df_1984, df_naip], ignore_index=True)
combined_csv = 'data/outputs/quercus_objects_combined.csv'
df_combined.to_csv(combined_csv, index=False)

print(f'Combined CSV: {len(df_combined):,} rows')
print('By source:')
print(df_combined['source'].value_counts().to_string())

## 15. Interactive Map
Orange = 1984 aerial segments, Green = NAIP segments. Toggle layers in top-right.

In [ ]:
import pandas as pd, folium, geopandas as gpd
import rasterio, numpy as np, io, base64, os
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from rasterio.crs import CRS
from shapely import wkt
from pathlib import Path
from PIL import Image
from IPython.display import display

os.makedirs('data/outputs', exist_ok=True)
df = pd.read_csv(combined_csv)
sample = df.sample(frac=0.10, random_state=42).reset_index(drop=True)
print(f'Plotting 10% sample: {len(sample):,} objects')

sample['geometry'] = sample['geojson'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(sample, geometry='geometry', crs='EPSG:32610').to_crs('EPSG:4326')
clat = gdf.geometry.centroid.y.median()
clon = gdf.geometry.centroid.x.median()

m = folium.Map(location=[clat,clon], zoom_start=15,
               tiles='Esri.WorldImagery', attr='Esri')
folium.TileLayer('OpenStreetMap', name='OSM').add_to(m)

def tif_to_overlay(tif_path, max_px=512):
    with rasterio.open(tif_path) as src:
        w,s,e,n = transform_bounds(src.crs,CRS.from_epsg(4326),*src.bounds)
        sc=min(1.0,max_px/src.width); nw=max(1,int(src.width*sc)); nh=max(1,int(src.height*sc))
        tr,_,_ = calculate_default_transform(src.crs,CRS.from_epsg(4326),
                     src.width,src.height,*src.bounds,dst_width=nw,dst_height=nh)
        nb=min(src.count,3)
        data=np.zeros((nb,nh,nw),dtype=np.float32)
        for i in range(1,nb+1):
            reproject(source=rasterio.band(src,i),destination=data[i-1],
                      src_transform=src.transform,src_crs=src.crs,
                      dst_transform=tr,dst_crs=CRS.from_epsg(4326),
                      resampling=Resampling.bilinear)
    mn,mx=data.min(),data.max()
    d8=((data-mn)/(mx-mn+1e-6)*255).astype(np.uint8)
    if nb==1: d8=np.vstack([d8,d8,d8])
    rgb=np.moveaxis(d8[:3],0,-1)
    alpha=np.where(rgb.sum(axis=-1)==0,0,200).astype(np.uint8)
    buf=io.BytesIO(); Image.fromarray(np.dstack([rgb,alpha]),'RGBA').save(buf,'PNG')
    b64=base64.b64encode(buf.getvalue()).decode()
    return 'data:image/png;base64,'+b64, [[s,w],[n,e]]

g1984=folium.FeatureGroup(name='1984 Aerials',show=True)
for tif in sorted(Path('data/georef').glob('*_georef.tif')):
    try:
        b64,bounds=tif_to_overlay(tif)
        folium.raster_layers.ImageOverlay(image=b64,bounds=bounds,opacity=0.75).add_to(g1984)
    except Exception as e: print('WARN',tif.name,e)
g1984.add_to(m)

gnaip=folium.FeatureGroup(name='NAIP',show=False)
for tif in sorted(Path('data/naip_outputs/naip_tifs').glob('*.tif')):
    try:
        b64,bounds=tif_to_overlay(tif)
        folium.raster_layers.ImageOverlay(image=b64,bounds=bounds,opacity=0.75).add_to(gnaip)
    except Exception as e: print('WARN',tif.name,e)
gnaip.add_to(m)

segs=folium.FeatureGroup(name='SAM Segments',show=True)
for _,row in gdf.iterrows():
    src=str(row.get('source',''))
    color='#52b788' if 'naip' in src else '#f4a261'
    ndvi=row.get('ndvi',None)
    tip=f"{src} | NDVI:{round(ndvi,3) if pd.notna(ndvi) else 'N/A'} | {int(row.get('area_m2',0))}m2"
    try:
        folium.GeoJson(row.geometry.__geo_interface__,
            style_function=lambda f,c=color:{'fillColor':c,'color':c,'weight':1,'fillOpacity':0.4},
            tooltip=tip).add_to(segs)
    except Exception: pass
segs.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
display(m)

## 16. Diagnostics

In [ ]:
import os, rasterio, numpy as np
from pathlib import Path
from PIL import Image
import geopandas as gpd
import matplotlib.pyplot as plt

os.makedirs('data/outputs', exist_ok=True)

raw     = sorted(Path('data/raw').glob('*.jpg'))
clipped = sorted(Path('data/clipped').glob('*.tif'))
georef  = sorted(Path('data/georef').glob('*_georef.tif'))
gj84    = sorted(Path('data/sam_outputs_1984').glob('*.geojson'))
naip_tifs=sorted(Path('data/naip_outputs/naip_tifs').glob('*.tif'))
naip_gj = sorted(Path('data/naip_outputs/sam_outputs').glob('*.geojson'))

fig, axes = plt.subplots(2, 3, figsize=(18,10))
fig.suptitle('QUERCUS v8 Diagnostics', fontsize=14)

def show(ax, path, title):
    try:
        if str(path).endswith(('.tif','.tiff')):
            with rasterio.open(path) as src:
                bands=[1,2,3] if src.count>=3 else [1,1,1]
                img=np.moveaxis(src.read(bands),0,-1)
                if img.max()>255: img=((img/img.max())*255).astype(np.uint8)
        else:
            img=np.array(Image.open(path).convert('RGB'))
        ax.imshow(img,cmap='gray')
    except Exception as e:
        ax.text(0.5,0.5,str(e),ha='center',va='center',transform=ax.transAxes,fontsize=8)
    ax.set_title(title,fontsize=9); ax.axis('off')

if raw:     show(axes[0,0],raw[0],     f'1. Raw\n{raw[0].name}')
if clipped: show(axes[0,1],clipped[0], f'2. Clipped\n{clipped[0].name}')
if georef:  show(axes[0,2],georef[0],  f'3. Georef\n{georef[0].name}')
if gj84:
    gdf_s=gpd.read_file(gj84[0])
    if georef:
        with rasterio.open(georef[0]) as src:
            bands=[1,2,3] if src.count>=3 else [1,1,1]
            img=np.moveaxis(src.read(bands),0,-1)
            if img.max()>255: img=((img/img.max())*255).astype(np.uint8)
        axes[1,0].imshow(img,cmap='gray')
    axes[1,0].set_title(f'4. SAM 1984\n{len(gdf_s)} segments',fontsize=9)
    axes[1,0].axis('off')
if naip_tifs: show(axes[1,1],naip_tifs[0],f'5. NAIP tile\n{naip_tifs[0].name}')

ax=axes[1,2]; ax.axis('off')
lines=['SUMMARY v8','',
       f'Raw downloaded:   {len(raw)}',
       f'Clipped:          {len(clipped)}',
       f'Georef tiles:     {len(georef)}',
       f'SAM 1984 GeoJSON: {len(gj84)}',
       f'NAIP tiles:       {len(naip_tifs)}',
       f'NAIP SAM GeoJSON: {len(naip_gj)}','']
if gj84:
    gdf2=gpd.read_file(gj84[0])
    lines.append(f'Segs/img (1984):  {len(gdf2)}')
if naip_gj:
    gdf3=gpd.read_file(naip_gj[0])
    lines.append(f'Segs/img (NAIP):  {len(gdf3)}')
ax.text(0.05,0.95,'\n'.join(lines),transform=ax.transAxes,fontsize=10,
        verticalalignment='top',fontfamily='monospace',
        bbox=dict(boxstyle='round',facecolor='lightyellow',alpha=0.5))
ax.set_title('Summary')

plt.tight_layout()
plt.savefig('data/outputs/diagnostics.png',dpi=120,bbox_inches='tight')
plt.show()
print('Saved: data/outputs/diagnostics.png')

## 17. D3 — K-Means Classification + Oak Extraction

In [ ]:
from quercus.classify import run_kmeans

labelled_csv, gpkg_path, cluster_summary = run_kmeans(
    combined_csv,
    output_dir='data/outputs',
    k=KMEANS_K,
    k_range=(2,12),
)
print(cluster_summary)

In [ ]:
from IPython.display import Image as IPImage
import pathlib
elbow = pathlib.Path('data/outputs/elbow_plot.png')
if elbow.exists(): display(IPImage(str(elbow)))

In [ ]:
from quercus.classify import extract_oak_clusters
from IPython.display import Image as IPImage

oak_gpkg, map_path = extract_oak_clusters(
    labelled_csv,
    output_dir='data/outputs',
    ndvi_min=0.3,
    area_min_m2=50,
    stability_min=0.85,
    top_n_clusters=2,
)
display(IPImage(str(map_path)))
print(f'Oak GeoPackage -> {oak_gpkg}')

## 18. Save Outputs to Google Drive

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/QUERCUS_v8_outputs'
os.makedirs(SAVE_DIR, exist_ok=True)

folders = ['data/raw','data/clipped','data/georef',
           'data/outputs','data/sam_outputs_1984','data/naip_outputs']
for folder in folders:
    if os.path.exists(folder):
        dst = os.path.join(SAVE_DIR, folder.replace('/','_'))
        shutil.copytree(folder, dst, dirs_exist_ok=True)
        size = sum(os.path.getsize(os.path.join(r,f))
                   for r,_,fs in os.walk(dst) for f in fs)
        print(f'Saved {folder}: {size/1e6:.0f} MB')
    else:
        print(f'Skip {folder}')
print(f'All saved -> {SAVE_DIR}')

## 19. Restore from Google Drive (new session)

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/QUERCUS_v8_outputs'

for src_name, dst in [
    ('data_raw',              'data/raw'),
    ('data_clipped',          'data/clipped'),
    ('data_georef',           'data/georef'),
    ('data_outputs',          'data/outputs'),
    ('data_sam_outputs_1984', 'data/sam_outputs_1984'),
    ('data_naip_outputs',     'data/naip_outputs'),
]:
    src = f'{DRIVE}/{src_name}'
    if os.path.exists(src):
        os.makedirs(dst, exist_ok=True)
        shutil.copytree(src, dst, dirs_exist_ok=True)
        n = sum(len(f) for _,_,f in os.walk(dst))
        print(f'Restored {dst}: {n} files')
    else:
        print(f'Not in Drive: {src_name}')

raw_paths        = sorted(Path('data/raw').glob('*.jpg'))
clipped_paths    = sorted(Path('data/clipped').glob('*.tif'))
georef_paths     = sorted(Path('data/georef').glob('*_georef.tif'))
objects_csv_1984 = Path('data/outputs/quercus_objects_1984.csv')
objects_csv_naip = Path('data/outputs/quercus_objects_naip.csv')
combined_csv     = Path('data/outputs/quercus_objects_combined.csv')
labelled_csv     = Path('data/outputs/quercus_objects_clustered.csv')

print(f'raw:{len(raw_paths)} clipped:{len(clipped_paths)} georef:{len(georef_paths)}')
print('Ready to continue from any step.')

## Outputs

| File | Description |
|------|-------------|
| `data/raw/` | Downloaded 1984 JPEG frames |
| `data/clipped/` | Border-clipped images |
| `data/georef/` | Georeferenced GeoTIFFs |
| `data/outputs/mosaic_1984.tif` | 1984 aerial mosaic |
| `data/sam_outputs_1984/` | SAM segments per 1984 frame |
| `data/outputs/quercus_objects_1984.csv` | 1984 objects + NDVI |
| `data/naip_outputs/naip_tifs/` | Downloaded NAIP GeoTIFFs |
| `data/naip_outputs/sam_outputs/` | SAM segments per NAIP tile |
| `data/outputs/quercus_objects_naip.csv` | NAIP objects + NDVI |
| `data/outputs/mosaic_naip.tif` | NAIP mosaic |
| `data/outputs/quercus_objects_combined.csv` | Combined 1984 + NAIP |
| `data/outputs/quercus_objects_clustered.csv` | K-Means labelled |
| `data/outputs/quercus_oaks.gpkg` | Probable oak candidates |
| `data/outputs/quercus_oak_map.png` | Oak detection map |

## Package structure

```
quercus/
    ingest/
        download.py       -- fetch_iiif_images (AOI filtering)
        preprocess.py     -- clip_scan_borders
        georeference.py   -- georeference_images, make_estimated_gcps
        mosaic.py         -- build_mosaic
    segment/
        sam_runner.py     -- run_sam_pipeline (CLAHE + dense prompts)
        naip_pipeline.py  -- fetch_naip_and_run_sam (AOI filtering)
        gee_ndvi.py       -- fetch_gee_ndvi (UTM->WGS84 corrected)
        csv_builder.py    -- build_objects_csv
    classify/
        kmeans.py         -- run_kmeans
        oak_extractor.py  -- extract_oak_clusters
    utils/
        aoi.py            -- load_aoi, bbox_intersects_aoi
```

Install: `pip install git+https://github.com/MayerT1/quercus.git`